In [1]:
!nvidia-smi

Sat Jun 13 07:18:58 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.126.16             Driver Version: 580.126.16     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-80GB          On  |   00000000:47:00.0 Off |                    0 |
| N/A   25C    P0             63W /  400W |       0MiB /  81920MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

In [2]:
%cd /workspace
!git clone -q https://github.com/openvla/openvla.git
%cd openvla
!pip install -q -e . 2>&1 | tail -5
print("OpenVLA repo installed")

/workspace


/usr/local/lib/python3.11/dist-packages/IPython/core/magics/osm.py:417: UserWarning: This is now an optional IPython functionality, setting dhist requires you to install the `pickleshare` library.
  self.shell.db['dhist'] = compress_dhist(dhist)[-100:]


/workspace/openvla

[notice] A new release of pip is available: 24.2 -> 26.1.2
[notice] To update, run: python -m pip install --upgrade pip
OpenVLA repo installed


In [3]:
# Clone & install LIBERO + all simulation deps
%cd /workspace
!git clone -q https://github.com/Lifelong-Robot-Learning/LIBERO.git
%cd LIBERO
!pip install -q -e . 2>&1 | tail -3
!pip install -q robosuite==1.4.1 bddl==1.0.1 future easydict matplotlib imageio cloudpickle gym 2>&1 | tail -3

# System libs for headless MuJoCo rendering
!apt-get update -qq && apt-get install -y -qq libgl1-mesa-glx libosmesa6 libglfw3 libglew-dev patchelf > /dev/null 2>&1
print("LIBERO + deps installed")

/workspace
/workspace/LIBERO

[notice] A new release of pip is available: 24.2 -> 26.1.2
[notice] To update, run: python -m pip install --upgrade pip

[notice] A new release of pip is available: 24.2 -> 26.1.2
[notice] To update, run: python -m pip install --upgrade pip
LIBERO + deps installed


In [4]:
!pip install -q "numpy<2" 2>&1 | tail -2
import numpy; print("numpy:", numpy.__version__)   # want 1.26.x (restart kernel if still 2.x)

[notice] A new release of pip is available: 24.2 -> 26.1.2
[notice] To update, run: python -m pip install --upgrade pip
numpy: 1.26.4


In [5]:
# OpenVLA is version-sensitive — pin these
!pip install -q transformers==4.40.1 tokenizers==0.19.1 timm==0.9.10 2>&1 | tail -2
import transformers, timm, tokenizers
print("transformers:", transformers.__version__, "| timm:", timm.__version__, "| tokenizers:", tokenizers.__version__)

[notice] A new release of pip is available: 24.2 -> 26.1.2
[notice] To update, run: python -m pip install --upgrade pip
transformers: 4.40.1 | timm: 0.9.10 | tokenizers: 0.19.1


In [6]:
!pip install -q "imageio[ffmpeg]" 2>&1 | tail -2   # save rollout videos as mp4
print("ffmpeg backend installed")

[notice] A new release of pip is available: 24.2 -> 26.1.2
[notice] To update, run: python -m pip install --upgrade pip
ffmpeg backend installed


In [7]:
# ★ OpenVLA REQUIRES flash attention (the default attn_implementation="flash_attention_2").
# Do NOT swap it to "eager" — that triggers a causal-mask off-by-one (287 vs 286) and breaks every action.
# Compiles ~10-20 min on A100; MAX_JOBS caps build parallelism to avoid OOM.
!MAX_JOBS=4 pip install -q flash-attn==2.5.5 --no-build-isolation
import flash_attn; print("flash-attn:", flash_attn.__version__)


[notice] A new release of pip is available: 24.2 -> 26.1.2
[notice] To update, run: python -m pip install --upgrade pip
flash-attn: 2.5.5


In [8]:
import os
os.environ["MUJOCO_GL"] = "osmesa"
os.environ["PYOPENGL_PLATFORM"] = "osmesa"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

In [9]:
import numpy; print("numpy:", numpy.__version__)
from libero.libero import benchmark
print("LIBERO OK:", list(benchmark.get_benchmark_dict().keys()))

numpy: 1.26.4


Do you want to specify a custom path for the dataset folder? (Y/N):  N


Initializing the default config file...
The following information is stored in the config file: /root/.libero/config.yaml
benchmark_root: /workspace/LIBERO/libero/libero
bddl_files: /workspace/LIBERO/libero/libero/./bddl_files
init_states: /workspace/LIBERO/libero/libero/./init_files
datasets: /workspace/LIBERO/libero/libero/../datasets
assets: /workspace/LIBERO/libero/libero/./assets
LIBERO OK: ['libero_spatial', 'libero_object', 'libero_goal', 'libero_90', 'libero_10', 'libero_100']


In [11]:
!pip install -q "protobuf<5" "tensorflow-metadata==1.14.0" "tensorflow-datasets==4.9.3"

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
wandb 0.27.2 requires protobuf!=5.28.0,!=5.29.0,<8,>4.21.0, but you have protobuf 3.20.3 which is incompatible.

[notice] A new release of pip is available: 24.2 -> 26.1.2
[notice] To update, run: python -m pip install --upgrade pip


In [13]:
 !pip install -q "wandb==0.16.6"


[notice] A new release of pip is available: 24.2 -> 26.1.2
[notice] To update, run: python -m pip install --upgrade pip


In [14]:
# Smoke test: 1 trial. Should run WITHOUT "Caught exception" and produce real actions.
%cd /workspace/openvla
!python experiments/robot/libero/run_libero_eval.py \
  --model_family openvla \
  --pretrained_checkpoint openvla/openvla-7b-finetuned-libero-spatial \
  --task_suite_name libero_spatial \
  --center_crop True \
  --num_trials_per_task 1 \
  --use_wandb False 2>&1 | tail -40

/workspace/openvla
 80%|████████  | 8/10 [27:38<06:35, 197.77s/it]
Task: pick up the black bowl on the stove and place it on the plate
Starting episode 1...
Saved rollout MP4 at path ./rollouts/2026_06_13/2026_06_13-07_35_16--episode=8--success=True--task=pick_up_the_black_bowl_on_the_stove_and_place_it_o.mp4
Success: True
# episodes completed so far: 8
# successes: 7 (87.5%)
Current task success rate: 1.0
Current total success rate: 0.875
[Warning]: datasets path /workspace/LIBERO/libero/libero/../datasets does not exist!
[Warning]: datasets path /workspace/LIBERO/libero/libero/../datasets does not exist!

 90%|█████████ | 9/10 [30:26<03:08, 188.58s/it]
Task: pick up the black bowl next to the plate and place it on the plate
Starting episode 1...
Saved rollout MP4 at path ./rollouts/2026_06_13/2026_06_13-07_35_16--episode=9--success=True--task=pick_up_the_black_bowl_next_to_the_plate_and_place.mp4
Success: True
# episodes completed so far: 9
# successes: 8 (88.9%)
Current task success

In [5]:
!apt-get update -qq && apt-get install -y -qq ffmpeg

debconf: delaying package configuration, since apt-utils is not installed
(Reading database ... 24632 files and directories currently installed.)
Preparing to unpack .../libatomic1_12.3.0-1ubuntu1~22.04.3_amd64.deb ...
Unpacking libatomic1:amd64 (12.3.0-1ubuntu1~22.04.3) over (12.3.0-1ubuntu1~22.04) ...
Preparing to unpack .../libubsan1_12.3.0-1ubuntu1~22.04.3_amd64.deb ...
Unpacking libubsan1:amd64 (12.3.0-1ubuntu1~22.04.3) over (12.3.0-1ubuntu1~22.04) ...
Preparing to unpack .../gcc-12-base_12.3.0-1ubuntu1~22.04.3_amd64.deb ...
Unpacking gcc-12-base:amd64 (12.3.0-1ubuntu1~22.04.3) over (12.3.0-1ubuntu1~22.04) ...
Setting up gcc-12-base:amd64 (12.3.0-1ubuntu1~22.04.3) ...
(Reading database ... 24632 files and directories currently installed.)
Preparing to unpack .../libstdc++6_12.3.0-1ubuntu1~22.04.3_amd64.deb ...
Unpacking libstdc++6:amd64 (12.3.0-1ubuntu1~22.04.3) over (12.3.0-1ubuntu1~22.04) ...
Setting up libstdc++6:amd64 (12.3.0-1ubuntu1~22.04.3) ...
(Reading database ... 24632 f

In [6]:
%cd /workspace/openvla/rollouts/2026_06_13/
!ffmpeg -i "rollout.mp4" -vf "fps=10,scale=320:-1" rollout.gif

/workspace/openvla/rollouts/2026_06_13
ffmpeg version 4.4.2-0ubuntu0.22.04.1 Copyright (c) 2000-2021 the FFmpeg developers
  built with gcc 11 (Ubuntu 11.2.0-19ubuntu1)
  configuration: --prefix=/usr --extra-version=0ubuntu0.22.04.1 --toolchain=hardened --libdir=/usr/lib/x86_64-linux-gnu --incdir=/usr/include/x86_64-linux-gnu --arch=amd64 --enable-gpl --disable-stripping --enable-gnutls --enable-ladspa --enable-libaom --enable-libass --enable-libbluray --enable-libbs2b --enable-libcaca --enable-libcdio --enable-libcodec2 --enable-libdav1d --enable-libflite --enable-libfontconfig --enable-libfreetype --enable-libfribidi --enable-libgme --enable-libgsm --enable-libjack --enable-libmp3lame --enable-libmysofa --enable-libopenjpeg --enable-libopenmpt --enable-libopus --enable-libpulse --enable-librabbitmq --enable-librubberband --enable-libshine --enable-libsnappy --enable-libsoxr --enable-libspeex --enable-libsrt --enable-libssh --enable-libtheora --enable-libtwolame --enable-libvidstab --

In [8]:
import os
os.environ["MUJOCO_GL"] = "egl"
os.environ["PYOPENGL_PLATFORM"] = "egl"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

In [ ]:
%cd /workspace/openvla
!python -u experiments/robot/libero/run_libero_eval.py \
  --model_family openvla \
  --pretrained_checkpoint openvla/openvla-7b-finetuned-libero-spatial \
  --task_suite_name libero_spatial \
  --center_crop True \
  --num_trials_per_task 10 \
  --seed 7 \
  --use_wandb False

/workspace/openvla


/usr/local/lib/python3.11/dist-packages/IPython/core/magics/osm.py:417: UserWarning: This is now an optional IPython functionality, setting dhist requires you to install the `pickleshare` library.
  self.shell.db['dhist'] = compress_dhist(dhist)[-100:]


2026-06-13 08:30:24.397304: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9261] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2026-06-13 08:30:24.397377: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:607] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2026-06-13 08:30:24.398164: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1515] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-06-13 08:30:24.403825: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-06-13 08:30:25.490857: W tensorflow/compiler/tf2